<a href="https://colab.research.google.com/github/prasertrak/large-language-models-from-scratch/blob/main/code_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Appendix A: Introduction to PyTorch (Part 2)

## A.9 Optimizing training performance with GPUs

### A.9.1 PyTorch computations on GPU devices

In [1]:
import torch

print(torch.__version__)

2.11.0+cu130


In [2]:
print(torch.cuda.is_available())

True


In [3]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

print(tensor_1 + tensor_2)

tensor([5., 7., 9.])


In [4]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")

print(tensor_1 + tensor_2)

tensor([5., 7., 9.], device='cuda:0')


In [5]:
tensor_1 = tensor_1.to("cpu")
print(tensor_1 + tensor_2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

ข้อผิดพลาด **`RuntimeError: Expected all tensors to be on the same device...`** เกิดจากคุณกำลังนำเทนเซอร์ 2 ตัวที่อยู่คนละอุปกรณ์ (Device) มาคำนวณร่วมกัน (ตัวหนึ่งอยู่บน CPU อีกตัวอยู่บน GPU/CUDA)

ในโค้ดของคุณ `tensor_1` ถูกย้ายไปที่ `"cpu"` แล้ว แต่ `tensor_2` ยังคงอยู่บน GPU (`cuda:0`)

### 🛠️ วิธีแก้ไข

คุณต้องย้ายเทนเซอร์ทั้งสองตัวให้อยู่บนอุปกรณ์เดียวกันก่อนนำมาบวกกันครับ โดยเลือกวิธีใดวิธีหนึ่งดังนี้:

In [6]:
tensor_1 = tensor_1.to("cuda")  # ย้าย tensor_1 กลับไปที่ GPU
tensor_2 = tensor_2.to("cuda")  # มั่นใจว่า tensor_2 อยู่บน GPU
print(tensor_1 + tensor_2)


tensor([5., 7., 9.], device='cuda:0')


In [7]:
# เช็คว่ามี GPU ไหม ถ้ามีใช้ cuda ถ้าไม่มีใช้ cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ย้ายเทนเซอร์ทุกตัวไปที่ device เดียวกัน
tensor_1 = tensor_1.to(device)
tensor_2 = tensor_2.to(device)

print(tensor_1 + tensor_2)


tensor([5., 7., 9.], device='cuda:0')


### A.9.2 Single-GPU training

In [9]:
'''X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])'''

# ข้อมูลสำหรับฝึกสอนโมเดล (Training Data)
# X_train: ตัวแปรอิสระ (Features) มี 5 ตัวอย่าง แต่ละตัวอย่างมี 2 ฟีเจอร์ (เช่น พิกัด X, Y)
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

# y_train: ตัวแปรตาม หรือ คำตอบ (Labels/Targets) สำหรับข้อมูลฝึกสอน มี 5 คำตอบ (0 หรือ 1) สอดคล้องกับ X_train
y_train = torch.tensor([0, 0, 0, 1, 1])

# ข้อมูลสำหรับทดสอบโมเดล (Testing Data)
# X_test: ตัวแปรอิสระ (Features) ที่โมเดลไม่เคยเห็นมาก่อน มี 2 ตัวอย่าง สำหรับใช้ประเมินประสิทธิภาพ
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

# y_test: คำตอบที่ถูกต้อง (Labels/Targets) สำหรับข้อมูลทดสอบ ใช้สำหรับเฉลยและตรวจคะแนนของโมเดล
y_test = torch.tensor([0, 1])


In [12]:
'''from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)'''

from torch.utils.data import Dataset

# สร้างคลาส Custom Dataset โดยสืบทอดคุณสมบัติมาจาก Dataset ของ PyTorch
class ToyDataset(Dataset):
    def __init__(self, X, y):
        # กำหนดค่าเริ่มต้น: เก็บข้อมูลตัวแปรอิสระ (Features) และคำตอบ (Labels) ไว้ในออบเจกต์
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        # ดึงข้อมูลตัวอย่าง (Sample) และคำตอบที่ตำแหน่ง index นั้นๆ ออกมา 1 ตัว
        one_x = self.features[index]
        one_y = self.labels[index]
        # ส่งคืนค่าเป็นคู่ (Tuple) ของข้อมูลและคำตอบสำหรับนำไปใช้งาน
        return one_x, one_y

    def __len__(self):
        # ส่งคืนจำนวนข้อมูลทั้งหมดที่มีอยู่ใน Dataset (เช็คจากขนาดแถวแรกของ Labels)
        return self.labels.shape[0]


# สร้างอินสแตนซ์ (Instance) ของ Dataset สำหรับชุดข้อมูลฝึกสอน (Training)
train_ds = ToyDataset(X_train, y_train)

# สร้างอินสแตนซ์ (Instance) ของ Dataset สำหรับชุดข้อมูลทดสอบ (Testing)
test_ds = ToyDataset(X_test, y_test)


In [13]:
'''from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=1,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=1
)'''

from torch.utils.data import DataLoader

# ล็อกค่าสุ่ม (Random Seed) เพื่อให้การสุ่มข้อมูลทุกครั้งที่รันโค้ดได้ผลลัพธ์เหมือนเดิม (เปรียบเทียบผลทดลองได้แม่นยำ)
torch.manual_seed(123)

# สร้าง DataLoader สำหรับชุดข้อมูลฝึกสอน (Training Set)
train_loader = DataLoader(
    dataset=train_ds,  # ระบุชุดข้อมูลต้นทางที่ต้องการนำมาโหลด (ในที่นี้คือ train_ds)
    batch_size=2,      # แบ่งข้อมูลออกเป็นกลุ่มย่อย (Batch) กลุ่มละ 2 ตัวอย่าง
    shuffle=True,      # สับไพ่/สลับลำดับข้อมูลแบบสุ่มในทุกๆ รอบการเทรน (Epoch) เพื่อลดการจำลำดับตัวอย่างของโมเดล
    num_workers=1,     # ใช้โปรเซสย่อย (Subprocess) 1 ตัวในการช่วยโหลดข้อมูลเข้า Memory เพื่อความรวดเร็ว
    drop_last=True     # ตัดข้อมูลกลุ่มสุดท้ายทิ้ง หากจำนวนข้อมูลที่เหลือมีน้อยกว่าขนาด batch_size ที่ตั้งไว้ (ป้องกัน Error ตอนคำนวณ)
)

# สร้าง DataLoader สำหรับชุดข้อมูลทดสอบ (Testing Set)
test_loader = DataLoader(
    dataset=test_ds,   # ระบุชุดข้อมูลต้นทางสำหรับทดสอบ (ในที่นี้คือ test_ds)
    batch_size=2,      # แบ่งข้อมูลออกเป็นกลุ่มย่อยกลุ่มละ 2 ตัวอย่างเช่นกัน
    shuffle=False,     # ห้ามสลับลำดับข้อมูลในการทดสอบ เพื่อให้สามารถวัดผลเรียงตามชุดข้อมูลจริงได้อย่างถูกต้อง
    num_workers=1      # ใช้โปรเซสย่อย 1 ตัวในการช่วยโหลดข้อมูล
)


In [14]:
'''class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits'''


# สร้างคลาสโครงข่ายประสาทเทียม (Neural Network) โดยสืบทอดคุณสมบัติมาจาก torch.nn.Module
class NeuralNetwork(torch.nn.Module):

    def __init__(self, num_inputs, num_outputs):
        # เรียกใช้คอนสตรัคเตอร์ของคลาสแม่ (torch.nn.Module) เพื่อตั้งค่าระบบภายในของ PyTorch
        super().__init__()

        # นิยามโครงสร้างชั้นเลเยอร์ (Layers) ต่างๆ เรียงต่อกันเป็นเส้นตรงด้วย torch.nn.Sequential
        self.layers = torch.nn.Sequential(
            # ชั้นซ่อนที่ 1 (1st hidden layer): รับฟีเจอร์ขาเข้าตามจำนวน num_inputs แล้วขยายเพิ่มเป็น 30 โนด (Neurons)
            torch.nn.Linear(num_inputs, 30),
            # ฟังก์ชันเปิดใช้งาน (Activation Function) แบบ ReLU เพื่อเพิ่มความเป็นเชิงเส้น (Non-linearity) ให้โมเดลเรียนรู้ pattern ที่ซับซ้อนได้
            torch.nn.ReLU(),
            # ชั้นซ่อนที่ 2 (2nd hidden layer): รับข้อมูล 30 โนดจากเลเยอร์ก่อนหน้า แล้วบีบขนาดย่อยลงเหลือ 20 โนด
            torch.nn.Linear(30, 20),
            # ฟังก์ชันเปิดใช้งาน ReLU ประมวลผลผลลัพธ์ของชั้นซ่อนที่ 2
            torch.nn.ReLU(),
            # ชั้นผลลัพธ์ (Output layer): รับข้อมูล 20 โนด แล้วแปลงค่าแปลงขนาดออกมาตามจำนวนคลาสคำตอบ (num_outputs) ที่ต้องการ
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        # กำหนดทิศทางการไหลของข้อมูล (Forward Pass): นำข้อมูลขาเข้า x ส่งผ่านเข้าเลเยอร์ทั้งหมดที่นิยามไว้
        logits = self.layers(x)
        # ส่งค่า Logits (ผลลัพธ์ดิบที่ยังไม่ได้แปลงเป็นความน่าจะเป็น) กลับออกไปเพื่อนำไปคำนวณ Loss ต่อไป
        return logits


In [16]:
'''import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # NEW
model.to(device) # NEW

# Note that the book originally used the following line, but the "model =" is redundant
# model = model.to(device) # NEW

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        features, labels = features.to(device), labels.to(device) # NEW
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation'''

import torch.nn.functional as F

# ล็อกค่าสุ่ม (Random Seed) เพื่อให้การสุ่มค่าน้ำหนักเริ่มต้นของโมเดลได้ผลลัพธ์เหมือนเดิมทุกครั้งที่รันโค้ด
torch.manual_seed(123)

# สร้างอินสแตนซ์ของโมเดล โดยกำหนดจำนวนข้อมูลขาเข้า (Inputs) เป็น 2 และจำนวนคลาสคำตอบ (Outputs) เป็น 2
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# ตรวจสอบและเลือกอุปกรณ์ที่จะใช้ประมวลผล: หากมี GPU ให้ใช้ "cuda" ถ้าไม่มีให้ใช้ "cpu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ย้ายยูนิตและค่าน้ำหนัก (Weights/Parameters) ทั้งหมดของโมเดลไปไว้บนอุปกรณ์ที่เลือก (CPU หรือ GPU)
model.to(device)

# นิยามตัวเพิ่มประสิทธิภาพ (Optimizer) แบบ SGD (Stochastic Gradient Descent)
# โดยส่งค่าพารามิเตอร์ของโมเดลให้คอยอัปเดต และกำหนดอัตราการเรียนรู้ (Learning Rate: lr) เป็น 0.5
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# กำหนดจำนวนรอบ (Epochs) ทั้งหมดที่จะให้โมเดลเรียนรู้ซ้ำจากชุดข้อมูลทั้งหมด
num_epochs = 3

# เริ่มต้นลูปการเทรนตามจำนวนรอบที่กำหนด
for epoch in range(num_epochs):

    # เปลี่ยนโหมดของโมเดลให้อยู่ใน "โหมดฝึกสอน" (Training Mode) เพื่อเปิดใช้งานฟังก์ชันอย่าง Dropout หรือ BatchNorm (ถ้ามี)
    model.train()

    # ดึงข้อมูลออกมาทีละกลุ่ม (Batch) จาก train_loader โดยระบุลำดับกลุ่มด้วย batch_idx
    for batch_idx, (features, labels) in enumerate(train_loader):

        # ย้ายข้อมูลฟีเจอร์และคำตอบเฉลย (Labels) ในรอบนั้นๆ ไปยังอุปกรณ์เดียวกันกับโมเดล (CPU หรือ GPU) ก่อนคำนวณ
        features, labels = features.to(device), labels.to(device)

        # ส่งข้อมูลเข้าโมเดลเพื่อทำ Forward Pass และรับค่าผลลัพธ์ดิบ (Logits) กลับมา
        logits = model(features)

        # คำนวณค่าความเสียหาย (Loss) ด้วย Cross Entropy ระหว่างผลลัพธ์ดิบจากโมเดลกับคำตอบที่ถูกต้อง
        loss = F.cross_entropy(logits, labels)

        # ล้างค่าเกรเดียนต์ (Gradients) หรือค่าความชันเก่าที่ค้างอยู่ในตัวแปรออกให้เป็นศูนย์ ป้องกันการสะสมทับของเดิม
        optimizer.zero_grad()

        # ทำ Backward Pass เพื่อคำนวณหาค่าความชัน (Gradients) ของพารามิเตอร์ต่างๆ ย้อนกลับตามค่า Loss ที่เกิดขึ้น
        loss.backward()

        # อัปเดตค่าน้ำหนัก (Weights) และค่า Bias ของโมเดลตามค่าเกรเดียนต์ที่คำนวณได้โดยใช้หลักการของ SGD
        optimizer.step()

        ### การพิมพ์บันทึกผล (LOGGING)
        # แสดงผลลัพธ์การฝึกสอน: ระบุรอบ (Epoch), กลุ่มข้อมูล (Batch), และค่าความเสียหาย (Loss) ณ ปัจจุบัน
        print(
            f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
            f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
            f" | Train/Val Loss: {loss:.2f}"
        )

    # เปลี่ยนโหมดของโมเดลให้อยู่ใน "โหมดประเมินผล" (Evaluation Mode) เพื่อปิดการฝึกสอน และเตรียมวัดผลโมเดล
    model.eval()
    # (สามารถเขียนโค้ดสำหรับประเมินผลลัพธ์หรือวัดความแม่นยำของโมเดลเพิ่มเติมตรงนี้ได้)


Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Train/Val Loss: 0.00


In [17]:
'''def compute_accuracy(model, dataloader, device):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        features, labels = features.to(device), labels.to(device) # New

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()'''

# นิยามฟังก์ชันสำหรับคำนวณค่าความแม่นยำ (Accuracy) ของโมเดล
def compute_accuracy(model, dataloader, device):

    # เปลี่ยนโหมดของโมเดลให้อยู่ใน "โหมดประเมินผล" (Evaluation Mode) เพื่อปิดการใช้งาน Dropout หรือ BatchNorm
    model = model.eval()

    # กำหนดตัวแปรเริ่มต้นสำหรับนับจำนวนตัวอย่างที่ทายถูก (เก็บค่าเป็นทศนิยมเพื่อป้องกันปัญหา Integer Division)
    correct = 0.0

    # กำหนดตัวแปรเริ่มต้นสำหรับนับจำนวนตัวอย่างทั้งหมดที่ถูกนำมาประเมิน
    total_examples = 0

    # วนลูปดึงข้อมูลทีละกลุ่ม (Batch) ออกมาจาก DataLoader
    for idx, (features, labels) in enumerate(dataloader):

        # ย้ายข้อมูลฟีเจอร์และคำตอบเฉลย (Labels) ไปยังอุปกรณ์ที่กำหนด (CPU หรือ GPU) ให้ตรงกับที่โมเดลอยู่
        features, labels = features.to(device), labels.to(device)

        # ใช้บริบท torch.no_grad() เพื่อปิดการคำนวณเกรเดียนต์/ความชัน ช่วยประหยัดหน่วยความจำ (Memory) และทำให้ประมวลผลเร็วขึ้น
        with torch.no_grad():
            # ส่งข้อมูลเข้าโมเดลเพื่อทำ Forward Pass และรับค่าผลลัพธ์ดิบ (Logits)
            logits = model(features)

        # แปลงผลลัพธ์ดิบ (Logits) ให้เป็นดัชนีของคลาสที่มีค่าสูงสุดในมิติที่ 1 (dim=1) ซึ่งก็คือ "คลาสที่โมเดลเลือกทาย"
        predictions = torch.argmax(logits, dim=1)

        # เปรียบเทียบคำตอบที่แท้จริง (labels) กับคำตอบที่โมเดลทาย (predictions) จะได้ผลลัพธ์เป็น Tensor ของค่าความจริง (True/False)
        compare = labels == predictions

        # รวมจำนวนที่ทายถูกทั้งหมดใน Batch นี้ (True = 1, False = 0) แล้วบวกสะสมเข้าตัวแปร correct
        correct += torch.sum(compare)

        # นับจำนวนตัวอย่างทั้งหมดใน Batch นี้จากการหาความยาวของ Tensor ผลลัพธ์เปรียบเทียบ แล้วบวกสะสมเข้าตัวแปร total_examples
        total_examples += len(compare)

    # คำนวณอัตราส่วนความแม่นยำ (correct / total_examples) แล้วใช้คำสั่ง .item() เพื่อแปลงค่าจาก PyTorch Tensor ออกมาเป็นตัวเลข Python ธรรมดา
    return (correct / total_examples).item()


In [19]:
# เรียกใช้งานฟังก์ชัน compute_accuracy เพื่อคำนวณความแม่นยำของชุดข้อมูลฝึกสอน (Training Set)
# โดยส่งตัวแปร model, train_loader และระบุอุปกรณ์ประมวลผล (device) ที่โมเดลอยู่ ณ ปัจจุบัน เข้าไปในฟังก์ชัน
compute_accuracy(model, train_loader, device=device)


1.0

In [20]:
# เรียกใช้งานฟังก์ชัน compute_accuracy เพื่อคำนวณความแม่นยำของชุดข้อมูลทดสอบ (Testing Set)
# โดยส่งตัวแปร model, test_loader และระบุอุปกรณ์ประมวลผล (device) เพื่อประเมินประสิทธิภาพของโมเดลกับข้อมูลใหม่ที่โมเดลไม่เคยเห็นตอนเทรน
compute_accuracy(model, test_loader, device=device)


1.0

### A.9.3 Training with multiple GPUs

See [DDP-script.py](DDP-script.py)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/12.webp" width="600px">
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/13.webp" width="600px">